# Reasoning models

MichAl Academy, unit 4.5.

Unit 4.3 measured that one transformer block does one hop of "find the thing
that the other thing points at", and that no amount of width buys a second one.
Depth is the budget for that kind of question, and depth is fixed once a model
is built.

This notebook measures the way round it. A model that writes its intermediate
results into the sequence reads them back on the next forward pass, so the same
fixed depth is spent again, once per token written. That is what chain of
thought does mechanically, and it is measurable on a model small enough to train
here.


In [ ]:
import copy
import time

import numpy as np
import torch
from torch import nn

torch.set_num_threads(1)

SLOTS = 16
QUERY, EQ, END = SLOTS, SLOTS + 1, SLOTS + 2
VOCAB = SLOTS + 3
PREFIX = SLOTS + 3               # the table, a marker, the starting slot, '='
MAXLEN = PREFIX + 8
STEPS = 3000


## The task

Sixteen slots. Each one holds a number, and that number names another slot.
Given a starting slot, follow the chain three times and report where you land.

A sequence is the sixteen slot contents, a marker, the starting slot, an `=`,
and then the answer. Two formats for that last part:

- **answer only**, one token: where the chain ends.
- **steps written out**, one token per hop: where the chain is after each one.

Tables are drawn fresh for every batch. Nothing repeats, so nothing can be
memorised, and a model that scores badly is failing at the task rather than
failing to remember it.


In [ ]:
def make_batch(n, hops, scratchpad, rng):
    """n sequences of the k-hop task, and the true chain for each of them."""
    table = rng.integers(0, SLOTS, (n, SLOTS))
    start = rng.integers(0, SLOTS, n)
    chain = [start]
    for _ in range(hops):
        chain.append(table[np.arange(n), chain[-1]])
    chain = np.stack(chain[1:], axis=1)
    written = chain if scratchpad else chain[:, -1:]
    seq = np.concatenate([
        table,
        np.full((n, 1), QUERY),
        start[:, None],
        np.full((n, 1), EQ),
        written,
        np.full((n, 1), END),
    ], axis=1)
    return seq, chain


for scratchpad in (False, True):
    seq, chain = make_batch(1, 3, scratchpad, np.random.default_rng(3))
    print("steps written out" if scratchpad else "answer only")
    print("  slots  ", " ".join(f"{v:2d}" for v in seq[0, :SLOTS]))
    print("  from slot", seq[0, SLOTS + 1], "the chain runs",
          " -> ".join(str(v) for v in chain[0]))
    print("  tokens ", seq[0])


## The model

The same causal transformer as unit 4.4: an embedding table, learned positions,
two blocks, and a mask so no position can see the tokens after it. Between the
runs below, only the number of blocks and the format of the target change.

The loss is applied to the answer tokens alone. The table is random, so asking
the model to predict it would be asking it to guess noise.


In [ ]:
class Block(nn.Module):
    def __init__(self, d, heads):
        super().__init__()
        self.heads = heads
        self.q, self.k, self.v = (nn.Linear(d, d) for _ in range(3))
        self.proj = nn.Linear(d, d)
        self.n1, self.n2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.mlp = nn.Sequential(nn.Linear(d, 4 * d), nn.ReLU(), nn.Linear(4 * d, d))

    def forward(self, x, mask):
        h = self.n1(x)
        B, L, d = h.shape
        dh = d // self.heads
        shape = (B, L, self.heads, dh)
        q = self.q(h).view(shape).transpose(1, 2)
        k = self.k(h).view(shape).transpose(1, 2)
        v = self.v(h).view(shape).transpose(1, 2)
        s = (q @ k.transpose(-2, -1)) / dh ** 0.5
        s = s.masked_fill(mask, float("-inf")).softmax(dim=-1)
        x = x + self.proj((s @ v).transpose(1, 2).reshape(B, L, d))
        return x + self.mlp(self.n2(x))


class Model(nn.Module):
    def __init__(self, blocks=2, d=64, heads=4):
        super().__init__()
        self.emb = nn.Embedding(VOCAB, d)
        self.pos = nn.Embedding(MAXLEN, d)
        self.blocks = nn.ModuleList([Block(d, heads) for _ in range(blocks)])
        self.norm = nn.LayerNorm(d)
        self.out = nn.Linear(d, VOCAB)
        self.register_buffer("mask", torch.triu(
            torch.ones(MAXLEN, MAXLEN, dtype=torch.bool), 1))

    def forward(self, x):
        L = x.shape[1]
        h = self.emb(x) + self.pos(torch.arange(L))
        for b in self.blocks:
            h = b(h, self.mask[:L, :L])
        return self.out(self.norm(h))


loss_fn = nn.CrossEntropyLoss()


def train(model, hops, scratchpad, steps, lr=1e-3, seed=0, n=64):
    rng = np.random.default_rng(seed)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for _ in range(steps):
        seq, _ = make_batch(n, hops, scratchpad, rng)
        x = torch.tensor(seq)
        opt.zero_grad()
        logits = model(x[:, :-1])
        target = x[:, 1:].clone()
        target[:, :PREFIX - 1] = -100          # score the answer tokens only
        loss_fn(logits.reshape(-1, VOCAB), target.reshape(-1)).backward()
        opt.step()
    return model


@torch.no_grad()
def emit(model, seq, tokens, temp=None, generator=None):
    """Let the model write its answer, one token at a time."""
    model.eval()
    ids = torch.tensor(seq[:, :PREFIX]) if isinstance(seq, np.ndarray) else seq
    for _ in range(tokens):
        last = model(ids)[:, -1]
        if temp is None:
            nxt = last.argmax(dim=-1, keepdim=True)
        else:
            nxt = torch.multinomial((last / temp).softmax(dim=-1), 1,
                                    generator=generator)
        ids = torch.cat([ids, nxt], dim=1)
    model.train()
    return ids


print(f"{sum(p.numel() for p in Model().parameters()):,} parameters, two blocks")
print(f"chance is {1 / SLOTS:.4f}")


## Writing the steps down

Four runs, the same width and the same number of training steps. The first two
are the comparison; the last two exist so the first two can be read at all.

- **3 hops, answer only.** Two blocks, three hops asked for in one pass.
- **3 hops, steps written out.** The same model, allowed to emit the chain.
- **1 hop, answer only.** If this fails, the training is broken and nothing
  else here means anything.
- **3 hops, answer only, three blocks and four blocks.** If these succeed, the
  limit is depth rather than the task being impossible. Three is the number the
  rule "one block per hop" predicts, so it is the one worth looking at.

Every score is taken on the same 4,000 sequences, so the numbers can be
compared with each other and the split further down recombines into the whole.


In [ ]:
N = 4000
started = time.time()
models = {}
for label, blocks, hops, scratchpad in (
        ("3 hops, answer only", 2, 3, False),
        ("3 hops, steps written out", 2, 3, True),
        ("1 hop, answer only", 2, 1, False),
        ("3 hops, answer only, 3 blocks", 3, 3, False),
        ("3 hops, answer only, 4 blocks", 4, 3, False)):
    torch.manual_seed(0)
    model = train(Model(blocks), hops, scratchpad, STEPS)
    models[label] = model
    seq, chain = make_batch(N, hops, scratchpad, np.random.default_rng(100))
    written = emit(model, seq, hops if scratchpad else 1)[:, -1].numpy()
    print(f"{label:32s} {(written == chain[:, -1]).mean():.4f}"
          f"   ({time.time() - started:.0f}s)")


The model that cannot answer in one pass answers perfectly when it may write the
chain down, and the deeper runs say the task was learnable all along. Depth is
what the direct run ran out of, and tokens bought it back.

Three blocks for three hops is what the one-block-per-hop rule predicts, and it
gets most of the way rather than all of it. The minimum depth an argument calls
for is not the depth a trained model comfortably works at.

## Where the direct model's score comes from

It does not fall to chance, and that number needs explaining before it is quoted.

A chain can arrive somewhere it does not leave. If a slot points at itself, or
the chain revisits a slot it has already visited, the third hop changes nothing
and the answer was already reachable in fewer. Those cases can be counted
exactly, and they set the score a pure shortcut would reach.


In [ ]:
seq, chain = make_batch(N, 3, False, np.random.default_rng(100))
answer = chain[:, -1]
got = emit(models["3 hops, answer only"], seq, 1)[:, -1].numpy() == answer

table, start = seq[:, :SLOTS], seq[:, SLOTS + 1]
one = table[np.arange(N), start]
two = table[np.arange(N), one]
short = (answer == two) | (answer == one) | (answer == start)
share = short.mean()

print(f"answerable in fewer than three hops      {share:.4f}")
print(f"a shortcut taking all of those, guessing the rest  "
      f"{share + (1 - share) / SLOTS:.4f}")
print(f"the model on those chains                {got[short].mean():.4f}")
print(f"the model on chains needing three hops   {got[~short].mean():.4f}")
print(f"recombined                               "
      f"{share * got[short].mean() + (1 - share) * got[~short].mean():.4f}")
print(f"measured overall                         {got.mean():.4f}")


The shortcut scores higher than the model does. On the chains that genuinely
need three hops the model reaches a little over twice chance, which is the same
faint signal unit 4.3.3 measured for one block on two hops: something picked up,
nothing answered.

## A model that is not certain

Everything above is a model that either cannot do the task or does it perfectly.
The next two questions, about training on its own attempts and about spending
more at the moment of asking, both need a model that is sometimes right.

Making the lookup harder does not produce one. At 64 slots instead of 16 the
model still scores 1.0000. What does produce one is less width, because each
emitted step is a lookup and a narrow model has less room to tell sixteen slots
apart.


In [ ]:
started = time.time()
for d in (8, 12, 16, 24):
    torch.manual_seed(0)
    m = train(Model(2, d=d), 5, True, 1500)
    seq, chain = make_batch(2000, 5, True, np.random.default_rng(100))
    acc = (emit(m, seq, 5)[:, -1].numpy() == chain[:, -1]).mean()
    print(f"d={d:2d}  {sum(p.numel() for p in m.parameters()):6,d} parameters"
          f"  {acc:.4f}   ({time.time() - started:.0f}s)")


Capability arrives abruptly rather than gradually. The model used below is the
`d=16` one, at 5 hops.


In [ ]:
torch.manual_seed(0)
narrow = train(Model(2, d=16), 5, True, 1500)
seq, chain = make_batch(2000, 5, True, np.random.default_rng(500))
greedy = (emit(narrow, seq, 5)[:, -1].numpy() == chain[:, -1]).mean()
print(f"greedy accuracy: {greedy:.4f}")


## Answering more than once

Sampling makes the model's answer vary. Ask the same question sixteen times and
take the answer that came up most often, and see whether the vote beats one
greedy answer.

This runs on the same 2,000 questions the greedy score above was measured on,
and every later arm in this notebook uses them too. An earlier version drew a
fresh 1,000 here and printed the 2,000-question baseline beside it: two
measurements wearing the costume of one comparison, which is the mistake this
whole unit is about. Nothing in that code looked wrong, because the stale number
was sitting in a variable rather than in the cell.


In [ ]:
answer = chain[:, -1]
g = torch.Generator().manual_seed(0)
prefix = torch.tensor(seq[:, :PREFIX])
draws = np.stack([emit(narrow, prefix, 5, temp=1.0, generator=g)[:, -1].numpy()
                  for _ in range(16)], axis=1)
for k in (1, 2, 4, 8, 16):
    winner = np.array([np.bincount(row, minlength=VOCAB).argmax()
                       for row in draws[:, :k]])
    print(f"{k:2d} samples: {(winner == answer).mean():.4f}")
print(f"one greedy answer: {greedy:.4f}   (same {len(seq):,} questions)")


Read the ladder from the bottom. **One sampled answer is worse than one greedy
answer**, because drawing at random sometimes draws a worse token. More votes
climb steadily from there and pass greedy somewhere between four and eight, and
sixteen finish 0.0275 above it.

So the vote does help, a little, and it is worth being exact about what it is
doing. This chain has exactly one correct route, so separate attempts have no
independent routes to agree on. What the vote recovers is the model's own most
likely answer, which greedy decoding only approximates: greedy takes the
likeliest token at each step in turn, which is not the same as the likeliest
answer overall. That is a real gain and a bounded one. It cannot exceed what the
model already believes, and unit 4.6.1 measured what this model's belief is worth
at this depth.

Self-consistency proper needs several sound routes, which tend to arrive at the
same answer while mistakes scatter. That is a different and larger effect than
the one above, and this task cannot show it.

**Worth keeping as a habit: before spending on repeated attempts, ask whether
the problem has more than one way through.** If it has only one, expect the
small version of this, not the large one.

## Training on its own attempts

Sample an attempt for each of 4,000 questions, keep the ones whose final answer
is right, and train on those. Nobody writes a trace and nobody grades one.

The check below is the one that matters: of the traces the filter keeps, how
many were right the whole way through rather than right at the end?


In [ ]:
seq, chain = make_batch(4000, 5, True, np.random.default_rng(900))
answer = chain[:, -1]
g = torch.Generator().manual_seed(1)
traces = emit(narrow, torch.tensor(seq[:, :PREFIX]), 5, temp=1.0, generator=g)
traces = torch.cat([traces, torch.full((4000, 1), END)], dim=1)

written = traces[:, PREFIX:PREFIX + 5].numpy()
right = written[:, -1] == answer
whole_way = (written == chain).all(axis=1)

print(f"attempts reaching the right answer:           {right.mean():.4f}")
print(f"of those, every intermediate step right:      {whole_way[right].mean():.4f}")
print(f"of the rest, every intermediate step right:   {whole_way[~right].mean():.4f}")


That last line has to be 0.0000: an attempt with every step right ends on the
right answer, so it cannot be in the rejected pile. It is there as a check on
the code above rather than as a finding.

The line above it is the finding. Most of what an answer-only filter keeps is
reasoning that went wrong somewhere and landed correctly anyway.


In [ ]:
ground = torch.tensor(np.concatenate(
    [seq[:, :PREFIX], chain, np.full((4000, 1), END)], axis=1))


def finetune(model, data, steps=400, lr=3e-4, seed=0):
    torch.manual_seed(seed)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    rng = np.random.default_rng(seed)
    for _ in range(steps):
        x = data[torch.tensor(rng.integers(0, len(data), 64))]
        opt.zero_grad()
        logits = model(x[:, :-1])
        target = x[:, 1:].clone()
        target[:, :PREFIX - 1] = -100
        loss_fn(logits.reshape(-1, VOCAB), target.reshape(-1)).backward()
        opt.step()
    return model


check, check_chain = make_batch(2000, 5, True, np.random.default_rng(500))
print(f"untouched                    {'':16s}{greedy:.4f}")
for label, data in (("unfiltered, every attempt", traces),
                    ("filtered, answer correct", traces[torch.tensor(right)]),
                    ("fresh ground truth", ground)):
    tuned = finetune(copy.deepcopy(narrow), data)
    acc = (emit(tuned, check, 5)[:, -1].numpy() == check_chain[:, -1]).mean()
    print(f"{label:28s} {len(data):5d} traces  {acc:.4f}")


Nothing moves. Every arm sits inside the noise, including training on correct
traces supplied outright, and the reason is the measurement two cells up: barely
a quarter of the kept traces are right the whole way through, so the filtered
training set is mostly wrong reasoning that happened to land.

A model has to be good enough for the filter to have something worth keeping
before it can bootstrap from itself. This one is not, and the null result is
reported rather than tuned away, because its cause is the thing the section
teaches.
